In [ ]:
# ! pip install google-cloud-aiplatform vertexai

In [ ]:
import vertexai
vertexai.init(project="ivanmkc-experimental-2-631260")

In [ ]:
# results

In [ ]:
from llm_auditor.claims import Claim, QueryAmbiguity
import yaml
from pathlib import Path
from dataclasses import asdict

claims_dir_path = Path("claims")

# Read the YAML file
with open(claims_dir_path / "claims_with_queries.yaml", "r") as f:
    claims_from_yaml = yaml.safe_load(f)

# Convert the list of dictionaries back to a list of Claim objects
claims = [Claim(**claim_data) for claim_data in claims_from_yaml]

## Use auto-rater to rate claims

In [ ]:
import dotenv
dotenv.load_dotenv()

from vertexai import generative_models

model_name = "gemini-2.5-pro"
model: generative_models.GenerativeModel = generative_models.GenerativeModel(
    model_name=model_name,
)


In [ ]:
# import importlib

# import autorater
# importlib.reload(autorater)

In [ ]:
import asyncio
from autorater import eval_generation_async

semaphore = asyncio.Semaphore(10)
results = await asyncio.gather(*[eval_generation_async(
    model=model,
    question=claim.query, 
    model_reply=claim.claim, 
    ground_truth=claim.context, 
    semaphore=semaphore) 
    for claim in claims]
    )

In [ ]:
import pandas as pd

pd.DataFrame([dict(question=claim.query, 
                   model_reply=claim.claim,
                   ground_truth=claim.context,
                   score=result.score,
                   reason=result.reason) 
              for (claim, result) in zip(claims, results)])

## Revise claims using critique-revise agent

In [ ]:
# import dotenv
# from google.adk.runners import InMemoryRunner
# from google.genai.types import Part
# from google.genai.types import UserContent
# from llm_auditor.agent import root_agent, critic_agent, llm_auditor
# import textwrap

In [ ]:
# auditor_runner = InMemoryRunner(agent=llm_auditor)

# def create_verification_prompt(claim: str) -> str:
#     return f"Verify this claim: {claim}"

# @alru_cache(maxsize=None)
# async def revise_claim(claim: str) -> str:
#     """
#     Revises a claim using a runner session.

#     Args:
#         claim: The claim string to be evaluated.

#     Returns:
#         The rewritten claim.
#     """
#     session = auditor_runner.session_service.create_session(
#         app_name=auditor_runner.app_name, user_id="test_user"
#     )
#     content = UserContent(parts=[Part(text=create_verification_prompt(claim))])
#     events = []
#     async for event in auditor_runner.run_async(
#         user_id=session.user_id, session_id=session.id, new_message=content
#     ):
#         events.append(event)

#     raw_text = events[-1].content.parts[0].text

#     return raw_text

#     # return prompt.CriticOutput.model_validate_json(raw_text)

In [ ]:
# response = await revise_claim(claim=claims[0].claim)
# response

In [ ]:
# from llm_auditor.sub_agents.reviser import agent
# importlib.reload(agent)

In [ ]:
import importlib

# import revise
# importlib.reload(revise)
# importlib.reload(ClaimReviser)

# import llm_auditor
# from llm_auditor import agent
# importlib.reload(agent)

In [ ]:
from revise import ClaimReviser
from tqdm.asyncio import tqdm
from llm_auditor import agent
import asyncio

llm_auditor = agent.create_llm_auditor(
    agent_name="llm_auditor_financial",
    critic_agent_name="critic_agent_financial",
    reviser_agent_name="reviser_agent_financial",
    rag_corpus_id="projects/169190568756/locations/us-central1/ragCorpora/1152921504606846976"
)
claim_reviser = ClaimReviser(llm_auditor=llm_auditor)

async def revise_claims_async(claims: list[Claim], max_concurrency: int = 5) -> list[str | None]:
    """
    Asynchronously evaluates a list of Claim objects with concurrency control
    and a progress bar.

    Args:
        claims: A list of Claim dataclass instances to evaluate.
        max_concurrency: The maximum number of concurrent evaluation tasks.

    Returns:
        A list of revised claim str's, or None if the revision failed for that claim.
    """
    semaphore = asyncio.Semaphore(max_concurrency)
    tasks = [claim_reviser.revise_claim_async(claim.claim, semaphore) for claim in claims]

    print(f"\nStarting asynchronous evaluation of {len(claims)} claims (max concurrency: {max_concurrency})...")
    # Use tqdm.asyncio.tqdm.gather for concurrent execution with a progress bar
    results = await tqdm.gather(*tasks, desc="Revising Claims")
    print("Asynchronous revision complete.")
    return results

In [ ]:
revised_claims = await revise_claims_async(claims=claims)

assert len(revised_claims) == len(claims)

In [ ]:
semaphore = asyncio.Semaphore(10)
revised_results = await asyncio.gather(*[eval_generation_async(
    model=model,
    question=claim.query, 
    model_reply=revised_claim, 
    ground_truth=claim.context, 
    semaphore=semaphore) 
    for claim, revised_claim in zip(claims, revised_claims)]
    )

## Gather results into a df

In [ ]:
df_results = pd.DataFrame([dict(question=claim.query, 
                   answer=claim.claim,
                   context=claim.context,
                   is_supported=claim.is_supported,
                   score=result.score,
                   reason=result.reason,
                   revised_answer=revised_claim,
                   revised_score=revised_result.score,
                   revised_reason=revised_result.reason,
                   ) 
              for (claim, result, revised_claim, revised_result) in zip(claims, results, revised_claims, revised_results)
              if claim.is_supported == claim.is_supported_after_rewriting and claim.source_contains_context
              ])

In [ ]:
df_results.to_csv("autorater_results.csv")

In [ ]:
# df_results = pd.read_csv("autorater_results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'autorater_results.csv'

In [ ]:
df_results.head()

In [ ]:
import seaborn as sns

In [ ]:
# To use the index, we first turn it into a column
df_reset = df_results.reset_index()

# Now, convert this to a long-form DataFrame for Seaborn
df_long = pd.melt(df_reset, id_vars=['index', 'is_supported'], value_vars=['score', 'revised_score'],
                  var_name='line_name', value_name='score_value')

print("Long-form DataFrame ready for plotting:")
print(df_long.head())

## Show interaction of original vs revised and its effect on scores, conditioned on is_supported

You can observe that after revision, the is_supported=False have been corrected, and hence have a high score when compared with unrevised answers.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_long,
    x='line_name',        # Use the reset index for the x-axis
    y='score_value',
    hue='is_supported',  # Different colors for 'line1_data' and 'line2_data'
    # style='is_supported'  # Different line styles for categories 'A' and 'B'
)

plt.xlabel('Row Index')
plt.ylabel('Y Values')
plt.legend(title='is_supported')
plt.grid(True)
plt.show()

# Autorater score histograms

Another way to view it.

As expected, the revised_score shifts all the scores to 5.0 since it's rewriting is_supported == False statements to True.

In [ ]:
g = sns.FacetGrid(data=df_long, col='line_name', hue="is_supported", col_wrap=1)
g.map(sns.histplot, 'score_value')

## Stats

In [ ]:
df_long.groupby(["line_name", "is_supported"])["score_value"].mean()

## Improvement of revised_score over score

Let's show that baseline gets a score of X then after revision we get a score of Y > X.
Where the score is based on this prompt.

In [ ]:
df_results["revised_score_is_higher"] = df_results["revised_score"] > df_results["score"]

In [ ]:
# df_results

This shows an overall improvement, irregardless of is_supported.

In [ ]:
len(df_results[df_results["revised_score_is_higher"]])/len(df_results)

Percentage same or improved for is_supported == True

8% improvement, which makes sense since the unrevised text was already is_supported == True to begin with.

In [ ]:
len(df_results[df_results["revised_score_is_higher"] & df_results["is_supported"]])/len(df_results[df_results["is_supported"]])

Percentage same or improved for is_supported == False.


This is what we really care about, which is the ability of critique-reviser agent to "correct" non-supported answers into supported answers.

In [ ]:
len(df_results[df_results["revised_score_is_higher"] & ~df_results["is_supported"]])/len(df_results[df_results["is_supported"]])

### Visual representation

This is harder to read and probably less useful

In [31]:
# g = sns.FacetGrid(data=df_results, hue="revised_score_is_higher")
# g.map(sns.histplot, 'is_supported')

sns.histplot(data=df_results, x="revised_score_is_higher", hue="is_supported")

NameError: name 'sns' is not defined

# Examine mistakes

In [30]:
df_results[df_results["revised_score_is_higher"] & ~df_results["is_supported"]]

NameError: name 'df_results' is not defined